# Synchrony 원본 로드와 기본 전처리

고객·거래 데이터를 불러와 날짜와 식별자를 정리하고, 결제수단·결측치·관측 기간을 확인합니다.
이 노트북을 출발점으로 고객별 월별 집계와 분류·회귀·군집 입력을 작성합니다.

**작업 전제:** 제공 기록에 수집 누락이 없고 거래·해지 정보는 기록된 날짜에 이용 가능합니다.
관측 기간 안의 거래 기록 부재는 거래 없음으로 처리합니다.

## 1. 경로 설정

원본 ZIP의 기본 위치는 저장소의 `data/raw/Synchrony/Datasets.zip`입니다.
다른 파일을 사용하려면 `SOURCE_ZIP` 또는 `SYNCHRONY_SOURCE_ZIP` 환경 변수를 변경합니다.

In [ ]:
from pathlib import Path
import os
import sys
import zipfile

import pandas as pd

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src/experiments/synchrony/data.py").is_file()
     and (path / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Cardops 저장소 안에서 노트북을 실행해 주세요.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SOURCE_ZIP = Path(os.environ.get(
    "SYNCHRONY_SOURCE_ZIP", PROJECT_ROOT / "data/raw/Synchrony/Datasets.zip"
)).expanduser()
PROCESSED_DIR = PROJECT_ROOT / "data/processed/Synchrony"

if not SOURCE_ZIP.is_file():
    raise FileNotFoundError(f"원본 ZIP을 준비하거나 SOURCE_ZIP을 변경해 주세요: {SOURCE_ZIP}")

print("저장소:", PROJECT_ROOT)
print("원본:", SOURCE_ZIP)
print("향후 정제본 저장 위치:", PROCESSED_DIR)

## 2. 원본 로드와 날짜 변환

기존 실험의 `read_source`를 재사용합니다. 발급일·해지일·거래일을 날짜형으로 변환하며,
고객 ID와 거래 ID의 중복, 거래의 고객 ID가 고객 테이블에 존재하는지 검사합니다.

해지일이 비어 있다는 이유로 고객을 삭제하거나 임의의 날짜를 채우지 않습니다.

In [ ]:
from src.experiments.synchrony.data import read_source

customers, transactions = read_source(SOURCE_ZIP)
with zipfile.ZipFile(SOURCE_ZIP) as archive:
    payment_codes = pd.read_csv(archive.open("Datasets/Payment Code.csv"))
    category_codes = pd.read_csv(archive.open("Datasets/Category Code.csv"))

display(pd.DataFrame([
    {"테이블": "고객", "행 수": len(customers), "열 수": len(customers.reset_index().columns)},
    {"테이블": "거래", "행 수": len(transactions), "열 수": len(transactions.columns)},
    {"테이블": "결제수단", "행 수": len(payment_codes), "열 수": len(payment_codes.columns)},
    {"테이블": "상품 범주", "행 수": len(category_codes), "열 수": len(category_codes.columns)},
]))
# 거래 테이블에는 read_source가 거래일에서 만든 month 열이 추가됩니다.
display(customers.head(), transactions.head())

## 3. 자료형과 결측치 확인

결측치를 일괄 삭제·대체하기 전에 각 열의 의미를 확인합니다.
고객의 발급일이 없는 경우와 해지일이 없는 경우는 별도로 해석합니다.

In [ ]:
def column_summary(frame):
    return pd.DataFrame({
        "자료형": frame.dtypes.astype(str),
        "결측 수": frame.isna().sum(),
        "결측 비율(%)": frame.isna().mean().mul(100).round(2),
    })

display(column_summary(customers.reset_index()))
display(column_summary(transactions))
display(payment_codes, category_codes)
display(transactions["Transaction_Type"].value_counts(dropna=False).rename("행 수"))

## 4. 거래 관측 기간과 월별 규모

한 행이 여러 거래를 합산한 기록일 수 있으므로 **행 수와 거래건수를 따로 표시**합니다.
아래 표는 결제수단 전체의 구매(`Sale`) 기록을 요약합니다.

In [ ]:
observation_start = transactions["Transaction_Date"].min()
observation_end = transactions["Transaction_Date"].max()
print(f"거래 관측 기간: {observation_start.date()} ~ {observation_end.date()}")

sales = transactions.loc[transactions["Transaction_Type"].eq("Sale")].copy()
monthly_sales = sales.groupby("month").agg(
    customers=("Customer_ID", "nunique"),
    rows=("Transaction_ID", "size"),
    transaction_count=("Number_of_Transactions", "sum"),
    transaction_amount=("Transaction_Amount", "sum"),
)
display(monthly_sales)

## 5. 대상 카드 구매 기록 확인

기존 실험에서 대상 카드로 사용한 `Payment_Code = 3`을 결제수단 표와 함께 확인합니다.
이 단계에서는 전체 고객을 미래 해지 여부로 걸러내지 않습니다.
예측용 데이터를 만들 때 각 기준일에 발급되어 있고 아직 해지되지 않은 고객을 선택합니다.

In [ ]:
CARD_PAYMENT_CODE = 3
display(payment_codes.loc[payment_codes["Payment_Code"].eq(CARD_PAYMENT_CODE)])
card_sales = sales.loc[sales["Payment_Code"].eq(CARD_PAYMENT_CODE)].copy()

display(pd.Series({
    "대상 카드 구매 고객 수": card_sales["Customer_ID"].nunique(),
    "대상 카드 구매 기록 행 수": len(card_sales),
    "대상 카드 구매 거래건수 합계": card_sales["Number_of_Transactions"].sum(),
}, name="집계"))

## 6. 다음 전처리 작업

1. 금액·거래건수·반품의 처리 규칙을 확정합니다.
2. 고객별 월별 집계와 기준일별 고객 상태를 만듭니다.
3. 기준일까지의 기록에서 모델 입력을 만들고 미래 기록에서 정답을 만듭니다.
   - 분류: 이후 30일·60일 내 기록된 카드 해지 여부
   - 회귀: 다음 달 대상 카드 구매 거래건수 또는 금액
   - 군집: 기준일까지의 최근성·빈도·금액과 이용 변화
4. 학습·검증·평가 기간을 정합니다. 전처리 중 학습이 필요한 변환은 학습 기간에 맞춰 적용합니다.

정답을 확인할 미래 기간이 아직 끝나지 않은 표본은 정답을 0으로 채우지 않습니다.
정제본 저장 경로는 위에서 지정한 `PROCESSED_DIR`을 사용합니다.

월별 스냅샷 구현은 [`src/experiments/synchrony/data.py`](../../src/experiments/synchrony/data.py)를 참고합니다.